# DBRepo Setup Template

This notebook template supports T2.1 and T2.2. The visible DBRepo database and table schemas were created manually; the notebook remains useful for documenting metadata payloads and executing semantic metadata updates once an API token is available.

- Owner A: document database/table metadata.
- Owner B: add semantic mappings to DBRepo metadata.
- Owner C: load data and add unit mappings.


In [6]:
from pathlib import Path
import os
import requests

DBREPO_BASE_URL = os.environ.get("DBREPO_BASE_URL", "https://test.dbrepo.tuwien.ac.at")
DBREPO_DATABASE_ID = os.environ.get("DBREPO_DATABASE_ID", "ed890fa1-154c-4a66-8529-4088c97f68db")
DBREPO_TOKEN = os.environ.get("DBREPO_TOKEN")

HEADERS = {"Content-Type": "application/json"}
if DBREPO_TOKEN:
    HEADERS["Authorization"] = f"Bearer {DBREPO_TOKEN}"

SCHEMA_SQL = Path("../docs/schema.sql").read_text(encoding="utf-8")
print(DBREPO_BASE_URL)
print(DBREPO_DATABASE_ID)
print(f"Loaded schema SQL: {len(SCHEMA_SQL)} characters")


https://test.dbrepo.tuwien.ac.at
ed890fa1-154c-4a66-8529-4088c97f68db
Loaded schema SQL: 2850 characters


## A / T2.1: Database and Table Metadata

Use the metadata below when creating the DBRepo database and tables. Adapt endpoint paths to the exact DBRepo API documentation/account setup.

In [7]:
database_metadata = {
    "database_id": "ed890fa1-154c-4a66-8529-4088c97f68db",
    "database_name": "no2_air_quality_classification_austria",
    "database_internal_name": "no2_air_quality_classification_austria_jlnb",
    "title": "NO2 Air Quality Classification across Austrian Monitoring Stations",
    "description": (
        "Hourly nitrogen dioxide (NO2) measurements from Austrian monitoring sampling points, "
        "sourced from the European Environment Agency data hub and normalised for a FAIR "
        "machine-learning experiment."
    ),
    "keywords": [
        "NO2",
        "air quality",
        "Austria",
        "European Environment Agency",
        "environmental monitoring",
        "machine learning",
        "FAIR data",
    ],
    "source_publisher": "European Environment Agency",
    "curated_by": "Data Stewardship 2026 Group 3",
    "source_url": "https://www.eea.europa.eu/en/datahub/datahubitem-view/778ef9f5-6293-4846-badd-56a29c70880d",
    "access_date": "2026-03-18",
}
database_metadata


{'database_id': 'ed890fa1-154c-4a66-8529-4088c97f68db',
 'database_name': 'no2_air_quality_classification_austria',
 'database_internal_name': 'no2_air_quality_classification_austria_jlnb',
 'title': 'NO2 Air Quality Classification across Austrian Monitoring Stations',
 'description': 'Hourly nitrogen dioxide (NO2) measurements from Austrian monitoring sampling points, sourced from the European Environment Agency data hub and normalised for a FAIR machine-learning experiment.',
 'keywords': ['NO2',
  'air quality',
  'Austria',
  'European Environment Agency',
  'environmental monitoring',
  'machine learning',
  'FAIR data'],
 'source_publisher': 'European Environment Agency',
 'curated_by': 'Data Stewardship 2026 Group 3',
 'source_url': 'https://www.eea.europa.eu/en/datahub/datahubitem-view/778ef9f5-6293-4846-badd-56a29c70880d',
 'access_date': '2026-03-18'}

## B / T2.2: Semantic Mapping Payload

The mapping below mirrors `docs/semantic-mapping.md`. Use it as the payload source when DBRepo table/column metadata endpoints are known.

In [8]:
semantic_mappings = [
    {"table": "sampling_points", "column": "sampling_point_code", "concept": "http://www.w3.org/ns/sosa/FeatureOfInterest"},
    {"table": "pollutants", "column": "pollutant_id", "concept": "http://www.w3.org/ns/sosa/observedProperty"},
    {"table": "pollutants", "column": "pollutant_name", "concept": "http://purl.obolibrary.org/obo/CHEBI_33101"},
    {"table": "measurement_units", "column": "unit_code", "concept": "http://qudt.org/schema/qudt/unit"},
    {"table": "aggregation_types", "column": "aggregation_code", "concept": "http://www.w3.org/ns/sosa/usedProcedure"},
    {"table": "observation_logs", "column": "observation_log_id", "concept": "http://www.w3.org/ns/prov#Entity"},
    {"table": "measurements", "column": "start_time", "concept": "http://www.w3.org/2006/time#hasBeginning"},
    {"table": "measurements", "column": "end_time", "concept": "http://www.w3.org/2006/time#hasEnd"},
    {"table": "measurements", "column": "result_time", "concept": "http://www.w3.org/ns/sosa/resultTime"},
    {"table": "measurements", "column": "value", "concept": "http://www.w3.org/ns/sosa/hasSimpleResult"},
]
semantic_mappings


[{'table': 'sampling_points',
  'column': 'sampling_point_code',
  'concept': 'http://www.w3.org/ns/sosa/FeatureOfInterest'},
 {'table': 'pollutants',
  'column': 'pollutant_id',
  'concept': 'http://www.w3.org/ns/sosa/observedProperty'},
 {'table': 'pollutants',
  'column': 'pollutant_name',
  'concept': 'http://purl.obolibrary.org/obo/CHEBI_33101'},
 {'table': 'measurement_units',
  'column': 'unit_code',
  'concept': 'http://qudt.org/schema/qudt/unit'},
 {'table': 'aggregation_types',
  'column': 'aggregation_code',
  'concept': 'http://www.w3.org/ns/sosa/usedProcedure'},
 {'table': 'observation_logs',
  'column': 'observation_log_id',
  'concept': 'http://www.w3.org/ns/prov#Entity'},
 {'table': 'measurements',
  'column': 'start_time',
  'concept': 'http://www.w3.org/2006/time#hasBeginning'},
 {'table': 'measurements',
  'column': 'end_time',
  'concept': 'http://www.w3.org/2006/time#hasEnd'},
 {'table': 'measurements',
  'column': 'result_time',
  'concept': 'http://www.w3.org/ns/

## API Execution Placeholders

Replace these endpoint paths with the concrete DBRepo REST API calls after login/API documentation is available. Keep robust error handling for every request.

In [9]:
def post_json(endpoint: str, payload: dict):
    url = f"{DBREPO_BASE_URL.rstrip('/')}/{endpoint.lstrip('/')}"
    response = requests.post(url, headers=HEADERS, json=payload, timeout=30)
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        raise RuntimeError(f"DBRepo request failed: {response.status_code} {response.text}") from exc
    return response.json() if response.content else None

# Example only. Replace with the real endpoint from DBRepo documentation:
# created_database = post_json('/api/database', database_metadata)
# print(created_database)
